# Order Insights Assignment

This notebook follows the assignment step by step using only pandas, matplotlib, seaborn and Python's built-in `pathlib`.

## Question 1: Project setup

The virtual environment is already created. In that environment, install packages with:

`python -m pip install -r requirements.txt`

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
DATA_PATH = Path("data") / "orders_raw.csv"
OUTPUT_DIR = Path("output")
CHARTS_DIR = OUTPUT_DIR / "charts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

## Question 2: Open the dataset

Load the CSV into a DataFrame called `df`, then inspect its first rows, shape, columns and data types.

In [ ]:
df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.dtypes)

## Question 3: Check data quality

Look for missing values, duplicate rows and the unique values in `region` and `category`.

In [ ]:
print("Missing values:")
print(df.isna().sum())

print("Duplicate rows:", df.duplicated().sum())

print("Regions:", sorted(df["region"].dropna().unique()))
print("Categories:", sorted(df["category"].dropna().unique()))

### Three issues noticed

1. Some important values are missing or written as text such as `unknown` and `not_available`.
2. Some order values are negative or zero.
3. Region and category labels contain extra spaces and mixed case.

## Questions 4 to 6: Clean columns, remove invalid records and write a function

The function below copies the raw data, cleans text and data types, removes duplicates and removes rows without essential order details. Rows with missing `delivery_days` are kept.

In [ ]:
def clean_orders(df):
    cleaned = df.copy()

    for column in ["region", "category"]:
        cleaned[column] = cleaned[column].astype("string").str.strip().str.title()

    cleaned["order_date"] = pd.to_datetime(cleaned["order_date"], errors="coerce")

    for column in ["quantity", "unit_price", "order_value"]:
        cleaned[column] = pd.to_numeric(cleaned[column], errors="coerce")

    essential_columns = [
        "order_id",
        "order_date",
        "region",
        "category",
        "quantity",
        "unit_price",
        "order_value",
    ]

    cleaned = cleaned.drop_duplicates()
    cleaned = cleaned.dropna(subset=essential_columns)
    cleaned = cleaned[
        (cleaned["quantity"] > 0)
        & (cleaned["unit_price"] > 0)
        & (cleaned["order_value"] > 0)
    ]

    return cleaned.reset_index(drop=True)

In [ ]:
cleaned_df = clean_orders(df)
print("Cleaned shape:", cleaned_df.shape)
cleaned_df.head()

## Question 7: Save a quality report

Save the cleaned data and a small report showing how many rows were removed.

In [ ]:
cleaned_df.to_csv(OUTPUT_DIR / "orders_cleaned.csv", index=False)

report = pd.DataFrame(
    [
        {
            "rows_before_cleaning": len(df),
            "rows_after_cleaning": len(cleaned_df),
            "rows_removed": len(df) - len(cleaned_df),
        }
    ]
)
report.to_csv(OUTPUT_DIR / "data_quality_report.csv", index=False)
report

## Question 8: Explore the cleaned data

Use `groupby()` for sales by region, `value_counts()` for categories and `crosstab()` for region versus category.

In [ ]:
print("Mean order value:", round(cleaned_df["order_value"].mean(), 2))
print("Median order value:", round(cleaned_df["order_value"].median(), 2))
print("Standard deviation:", round(cleaned_df["order_value"].std(), 2))

In [ ]:
sales_by_region = cleaned_df.groupby("region")["order_value"].sum().sort_values(ascending=False)
sales_by_region

In [ ]:
cleaned_df["category"].value_counts()

In [ ]:
pd.crosstab(cleaned_df["region"], cleaned_df["category"])

## Question 9: Draw three charts

Each chart has a title, axis labels and is saved as a PNG file in `output/charts`.

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(cleaned_df["order_value"], bins=20)
plt.title("Histogram of Order Values")
plt.xlabel("Order Value")
plt.ylabel("Number of Orders")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "order_value_histogram.png")
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x=sales_by_region.index, y=sales_by_region.values)
plt.title("Sales by Region")
plt.xlabel("Region")
plt.ylabel("Total Sales")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "sales_by_region.png")
plt.show()

In [ ]:
scatter_df = cleaned_df.dropna(subset=["delivery_days", "order_value"])

plt.figure(figsize=(8, 5))
sns.scatterplot(data=scatter_df, x="delivery_days", y="order_value")
plt.title("Order Value vs Delivery Days")
plt.xlabel("Delivery Days")
plt.ylabel("Order Value")
plt.tight_layout()
plt.savefig(CHARTS_DIR / "order_value_vs_delivery_days.png")
plt.show()

## Question 10: Dashboard

Run `python -m streamlit run app.py` from the project folder to open the dashboard.